# Referral Bypass Detection - Booking-Centric Analysis

## tl;dr

This notebook turns the Step 1 dataset guide into a practical Step 2 detection workflow.

- The main business question is: **did a referred lead later book, but the booking was not honored as REFERRAL?**
- The core rule follows the manager discussion: when multiple referrals exist, match each booking to the **latest referral submitted before that booking**.
- The workflow keeps two views:
  - **Same-enquiry check:** the referral enquiry itself booked, but the booking source was not REFERRAL.
  - **Booking-centric bypass check:** a booking had at least one prior referral for the same lead, but the booking/enquiry source was not REFERRAL.
- Results are exported as CSV files for manager review or Tableau use.


## 1. Context & Workflow Norms

External workflow references support three simple principles for this task:

1. **Attribution window:** referral programs normally define a time period where a later purchase can still be credited to a referral.
2. **Attribution model:** when multiple referrals happen, platforms often use first-touch or last-touch rules. Here we use **last prior referral before booking** because the manager confirmed that multiple referrals can happen and we should proceed with that rule.
3. **Investigation flag:** the output is not a legal/finance final decision. It is a prioritized review list showing where referral credit may have been bypassed.

Reference examples used for method sanity check:
- [Refersion](https://support.refersion.com/en/articles/1064022-what-is-the-attribution-window): attribution window and last-touch click attribution.
- [Rewardful](https://www.rewardful.com/first-or-last-touch-attribution): first-touch vs last-touch affiliate attribution.
- [ReferralCandy](https://www.referralcandy.com/blog/referral-attribution): referral attribution needs clear windows, order-level tagging, and consistent rules.
- [Unit21](https://www.unit21.ai/trust-safety-dictionary/referral-fraud): referral abuse/fraud work often uses rule checks, link analysis, and manual review flags.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)

## 2. Setup & Data Loading

The notebook can run from the project root or from the `code` folder. It reads raw tables from `data/raw` and writes analysis outputs to `data/derived`.


In [ ]:
# Locate project folders
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "code" else CURRENT_DIR

SOURCE_DIR_CANDIDATES = [
    PROJECT_ROOT / "data" / "raw",
    CURRENT_DIR / "data" / "raw",
    CURRENT_DIR.parent / "data" / "raw",
]

SOURCE_DIR = next((p.resolve() for p in SOURCE_DIR_CANDIDATES if p.exists()), None)
if SOURCE_DIR is None:
    searched = "\n".join(str(p.resolve()) for p in SOURCE_DIR_CANDIDATES)
    raise FileNotFoundError(f"Cannot find data/raw. Current directory: {CURRENT_DIR}\nSearched:\n{searched}")

OUTPUT_DIR = PROJECT_ROOT / "data" / "derived"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(SOURCE_DIR.glob("*.csv"))
if len(csv_files) < 2:
    raise FileNotFoundError(f"Expected at least two CSV files in {SOURCE_DIR}, found {len(csv_files)}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SOURCE_DIR}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print("CSV files found:")
for f in csv_files:
    print(f"- {f.name}")


In [ ]:
# Load the two CSVs and detect their role by columns
loaded_tables = {f.name: pd.read_csv(f, dtype='string', keep_default_na=False, na_values=['']) for f in csv_files}

referral_file = None
booking_file = None
for file_name, table in loaded_tables.items():
    columns = set(table.columns)
    if {'referral_enquiry_id', 'referral_submitted_on', 'referrer_user_id'}.issubset(columns):
        referral_file = file_name
    if {'booking_id', 'bookingDate', 'booking_source'}.issubset(columns):
        booking_file = file_name

if referral_file is None or booking_file is None:
    raise ValueError('Could not identify referral and booking files from column names.')

referrals = loaded_tables[referral_file]
bookings = loaded_tables[booking_file]

print(f'Referral file: {referral_file} -> {referrals.shape[0]:,} rows, {referrals.shape[1]} columns')
print(f'Booking file: {booking_file} -> {bookings.shape[0]:,} rows, {bookings.shape[1]} columns')

## 3. Data Preparation & Quality Controls

This section standardizes dates, IDs, sources, and amount fields. It also checks required columns before any matching.

In [ ]:
REQUIRED_REFERRAL_COLUMNS = [
    'referral_enquiry_id', 'referral_submitted_on', 'source', 'project_id', 'projectName',
    'referrer_user_id', 'referrer_name', 'lead_id', 'lead_name', 'lead_hash_id',
    'mobileNumber', 'email', 'booking_on_referral_enquiry_id', 'booking_on_referral_source'
]

REQUIRED_BOOKING_COLUMNS = [
    'booking_id', 'bookingDate', 'booking_created_on', 'bookingStatus', 'booking_source',
    'agreementValue', 'enquiry_id', 'enquiry_source', 'lead_id', 'project_id', 'projectName',
    'lead_name', 'lead_hash_id', 'mobileNumber', 'email'
]

missing_referral_cols = [c for c in REQUIRED_REFERRAL_COLUMNS if c not in referrals.columns]
missing_booking_cols = [c for c in REQUIRED_BOOKING_COLUMNS if c not in bookings.columns]
if missing_referral_cols or missing_booking_cols:
    raise ValueError(f'Missing columns. Referrals: {missing_referral_cols}; Bookings: {missing_booking_cols}')

# Parse mixed timestamp formats safely across pandas versions
def parse_mixed_datetime(series):
    try:
        return pd.to_datetime(series, errors='coerce', format='mixed')
    except TypeError:
        return pd.to_datetime(series, errors='coerce', infer_datetime_format=True)

# Normalize source labels so NULL/blank is handled as not referral
def normalize_source(series):
    return series.fillna('').astype('string').str.strip().str.upper().replace({'': '<NULL>', 'NULL': '<NULL>'})

# Keep IDs as strings to avoid float-int-string conversion issues
def clean_id(series):
    return series.fillna('').astype('string').str.strip().replace({'': pd.NA, 'NULL': pd.NA})

referrals = referrals.assign(
    referral_enquiry_id=clean_id(referrals['referral_enquiry_id']),
    referral_submitted_at=parse_mixed_datetime(referrals['referral_submitted_on']),
    referral_source_norm=normalize_source(referrals['source']),
    referral_project_id=clean_id(referrals['project_id']),
    referral_lead_id=clean_id(referrals['lead_id']),
    booking_on_referral_enquiry_id_clean=clean_id(referrals['booking_on_referral_enquiry_id']),
    booking_on_referral_source_norm=normalize_source(referrals['booking_on_referral_source']),
    booking_on_referral_date_parsed=parse_mixed_datetime(referrals['booking_on_referral_date']) if 'booking_on_referral_date' in referrals.columns else pd.NaT,
)

bookings = bookings.assign(
    booking_id=clean_id(bookings['booking_id']),
    enquiry_id=clean_id(bookings['enquiry_id']),
    bookingDate_parsed=parse_mixed_datetime(bookings['bookingDate']),
    booking_created_on_parsed=parse_mixed_datetime(bookings['booking_created_on']),
    booking_timeline_at=lambda df: df['bookingDate_parsed'].fillna(df['booking_created_on_parsed']),
    booking_source_norm=normalize_source(bookings['booking_source']),
    enquiry_source_norm=normalize_source(bookings['enquiry_source']),
    booking_project_id=clean_id(bookings['project_id']),
    booking_lead_id=clean_id(bookings['lead_id']),
    agreementValue_num=pd.to_numeric(bookings['agreementValue'], errors='coerce'),
)

bookings = bookings.assign(
    booking_marked_referral=lambda df: (df['booking_source_norm'].eq('REFERRAL') | df['enquiry_source_norm'].eq('REFERRAL'))
)

quality_summary = pd.DataFrame([
    {'check': 'Referral rows', 'value': len(referrals)},
    {'check': 'Booking rows', 'value': len(bookings)},
    {'check': 'Referral rows with parsed submitted date', 'value': referrals['referral_submitted_at'].notna().sum()},
    {'check': 'Booking rows with usable booking timeline', 'value': bookings['booking_timeline_at'].notna().sum()},
    {'check': 'Unique referral leads', 'value': referrals['referral_lead_id'].nunique(dropna=True)},
    {'check': 'Unique booking leads', 'value': bookings['booking_lead_id'].nunique(dropna=True)},
    {'check': 'Referral duplicate enquiry IDs', 'value': referrals['referral_enquiry_id'].duplicated().sum()},
    {'check': 'Booking duplicate booking IDs', 'value': bookings['booking_id'].duplicated().sum()},
])

display(quality_summary)

## 4. Workflow A - Same Referral Enquiry Check

This checks the easiest case first: the referral enquiry itself has a booking. If that booking exists but is not marked as REFERRAL, it is an attribution anomaly.

In [ ]:
same_enquiry = referrals.loc[referrals['booking_on_referral_enquiry_id_clean'].notna(), [
    'referral_enquiry_id', 'referral_submitted_at', 'referral_enquiry_status', 'stage', 'source',
    'referralSource', 'project_id', 'projectName', 'referrer_user_id', 'referrer_name',
    'lead_id', 'lead_name', 'lead_hash_id', 'booking_on_referral_enquiry_id_clean',
    'booking_on_referral_date_parsed', 'booking_on_referral_status', 'booking_on_referral_source',
    'booking_on_referral_source_norm'
]].rename(columns={
    'project_id': 'referral_project_id_original',
    'lead_id': 'referral_lead_id_original',
    'booking_on_referral_enquiry_id_clean': 'booking_on_referral_enquiry_id',
})

same_enquiry = same_enquiry.assign(
    own_enquiry_honored=lambda df: df['booking_on_referral_source_norm'].eq('REFERRAL'),
    own_enquiry_status=lambda df: np.where(df['booking_on_referral_source_norm'].eq('REFERRAL'), 'Own Enquiry Honored', 'Own Enquiry Not Honored')
)

same_enquiry_anomalies = same_enquiry.loc[~same_enquiry['own_enquiry_honored']].sort_values('booking_on_referral_date_parsed')

same_enquiry_summary = same_enquiry.groupby('own_enquiry_status', dropna=False).agg(
    referral_enquiry_count=('referral_enquiry_id', 'nunique'),
    lead_count=('referral_lead_id_original', 'nunique')
).reset_index()

same_enquiry.to_csv(OUTPUT_DIR / 'same_referral_enquiry_honor_status.csv', index=False)
same_enquiry_anomalies.to_csv(OUTPUT_DIR / 'same_enquiry_attribution_anomalies.csv', index=False)
same_enquiry_summary.to_csv(OUTPUT_DIR / 'same_enquiry_honor_summary.csv', index=False)

print('Same-enquiry summary:')
display(same_enquiry_summary)
print(f'Same-enquiry anomalies exported: {len(same_enquiry_anomalies):,} rows')
display(same_enquiry_anomalies.head(10))

## 5. Workflow B - Booking-Centric Bypass Detection

This is the main workflow. It starts from each booking, finds prior referrals for the same lead, and keeps the latest prior referral as the attribution candidate.

Current matching rule:

> For each booking, match to the latest referral from the same lead where `referral_submitted_on <= bookingDate`.

If the booking has a prior referral but neither `booking_source` nor `enquiry_source` is REFERRAL, it becomes a suspected bypass candidate.

In [ ]:
referral_match_base = referrals.loc[
    referrals['referral_lead_id'].notna() & referrals['referral_submitted_at'].notna(),
    [
        'Client name', 'referral_enquiry_id', 'referral_submitted_at', 'referral_enquiry_status',
        'stage', 'referral_source_norm', 'referralSource', 'referralCode', 'relationToReferrer',
        'sourcePlatform', 'referral_project_id', 'projectName', 'project_crm_id',
        'referrer_user_id', 'referrer_name', 'referrer_crm_id', 'referrer_user_type',
        'referral_lead_id', 'lead_name', 'lead_hash_id', 'mobileNumber', 'email'
    ]
].rename(columns={
    'Client name': 'client_name',
    'projectName': 'referral_project_name',
    'lead_name': 'referral_lead_name',
    'lead_hash_id': 'referral_lead_hash_id',
    'mobileNumber': 'referral_mobile_encrypted',
    'email': 'referral_email_encrypted',
})

booking_match_base = bookings.loc[
    bookings['booking_lead_id'].notna() & bookings['booking_timeline_at'].notna(),
    [
        'Client name', 'booking_id', 'booking_crm_id', 'booking_timeline_at', 'bookingDate_parsed',
        'booking_created_on_parsed', 'bookingStatus', 'booking_source', 'booking_source_norm',
        'unitNumber', 'agreementValue_num', 'booking_user_id', 'enquiry_id', 'enquiry_source',
        'enquiry_source_norm', 'referralSource', 'enquiry_created_on', 'enquiry_user_id',
        'booking_lead_id', 'booking_project_id', 'projectName', 'lead_name', 'lead_hash_id',
        'mobileNumber', 'email', 'property_id', 'booking_marked_referral'
    ]
].rename(columns={
    'Client name': 'client_name',
    'projectName': 'booking_project_name',
    'lead_name': 'booking_lead_name',
    'lead_hash_id': 'booking_lead_hash_id',
    'mobileNumber': 'booking_mobile_encrypted',
    'email': 'booking_email_encrypted',
    'referralSource': 'booking_referralSource',
})

booking_referral_candidates = booking_match_base.merge(
    referral_match_base,
    left_on=['client_name', 'booking_lead_id'],
    right_on=['client_name', 'referral_lead_id'],
    how='left',
    suffixes=('', '_referral')
)

booking_referral_candidates = booking_referral_candidates.loc[
    booking_referral_candidates['referral_enquiry_id'].notna()
    & (booking_referral_candidates['referral_submitted_at'] <= booking_referral_candidates['booking_timeline_at'])
]

prior_referral_counts = booking_referral_candidates.groupby('booking_id').agg(
    prior_referral_count=('referral_enquiry_id', 'nunique'),
    prior_referrer_count=('referrer_user_id', 'nunique'),
    prior_project_count=('referral_project_id', 'nunique'),
    earliest_prior_referral_at=('referral_submitted_at', 'min'),
    latest_prior_referral_at=('referral_submitted_at', 'max'),
).reset_index()

latest_prior_referral_match = (
    booking_referral_candidates
    .sort_values(['booking_id', 'referral_submitted_at', 'referral_enquiry_id'])
    .drop_duplicates(subset=['booking_id'], keep='last')
    .merge(prior_referral_counts, on='booking_id', how='left')
)

latest_prior_referral_match = latest_prior_referral_match.assign(
    days_between_referral_and_booking=lambda df: (df['booking_timeline_at'] - df['referral_submitted_at']).dt.total_seconds() / 86400,
    same_project=lambda df: df['booking_project_id'].eq(df['referral_project_id']),
    case_type=lambda df: np.where(df['booking_marked_referral'], 'Honored Referral Booking', 'Suspected Referral Bypass'),
    bypass_reason=lambda df: np.where(
        df['booking_marked_referral'],
        'Prior referral found and booking/enquiry source is marked REFERRAL.',
        'Prior referral found before booking, but booking/enquiry source is not REFERRAL.'
    ),
    match_rule='Latest referral before booking by lead_id',
    needs_manual_review=lambda df: (
        df['prior_referral_count'].gt(1) | df['prior_referrer_count'].gt(1) | df['prior_project_count'].gt(1)
    )
)

export_columns = [
    'client_name', 'case_type', 'bypass_reason', 'match_rule', 'needs_manual_review',
    'booking_id', 'booking_crm_id', 'booking_timeline_at', 'bookingStatus', 'booking_source',
    'booking_source_norm', 'enquiry_id', 'enquiry_source', 'enquiry_source_norm',
    'booking_marked_referral', 'agreementValue_num', 'booking_user_id',
    'booking_lead_id', 'booking_lead_name', 'booking_lead_hash_id', 'booking_project_id',
    'booking_project_name', 'referral_enquiry_id', 'referral_submitted_at',
    'days_between_referral_and_booking', 'referral_enquiry_status', 'stage', 'referralSource',
    'referralCode', 'sourcePlatform', 'referral_project_id', 'referral_project_name',
    'same_project', 'referrer_user_id', 'referrer_name', 'referrer_user_type',
    'prior_referral_count', 'prior_referrer_count', 'prior_project_count',
    'earliest_prior_referral_at', 'latest_prior_referral_at'
]

booking_referral_match_cases = latest_prior_referral_match[export_columns].sort_values(['case_type', 'booking_timeline_at'])
suspected_bypass_cases = booking_referral_match_cases.loc[
    booking_referral_match_cases['case_type'].eq('Suspected Referral Bypass')
].sort_values('booking_timeline_at')

booking_referral_match_cases.to_csv(OUTPUT_DIR / 'booking_referral_match_cases.csv', index=False)
suspected_bypass_cases.to_csv(OUTPUT_DIR / 'suspected_bypass_cases.csv', index=False)

print(f'Bookings with at least one prior referral: {booking_referral_match_cases["booking_id"].nunique():,}')
print(f'Suspected bypass cases: {suspected_bypass_cases["booking_id"].nunique():,}')
display(suspected_bypass_cases)

## Overlap Check: Booking-Level Bypass vs Same-Enquiry Attribution Anomalies

Before reporting the two findings together, this section checks whether the `4` suspected booking-level bypass candidates overlap with the `28` same-enquiry attribution anomalies.

The overlap is checked at three levels:

1. `lead_id` level: whether the same referred lead appears in both findings.
2. `referral_enquiry_id` level: whether the referral enquiry used in the booking-level match appears in the same-enquiry anomaly table.
3. Strict booking / enquiry ID level: whether the suspected booking itself matches the same-enquiry anomaly booking ID or enquiry ID.

This prevents double-counting and clarifies whether the two findings are separate issues or the same issue viewed from different routes.

In [ ]:
# Overlap check between the 4 booking-level cases and the 28 same-enquiry anomalies
# The goal is to avoid double-counting before writing the final interpretation.

suspected_bypass_review = suspected_bypass_cases.copy()
same_enquiry_anomaly_review = same_enquiry_anomalies.copy()

# 1) Lead-level overlap
suspected_lead_ids = set(suspected_bypass_review['booking_lead_id'].dropna().astype('string'))
same_enquiry_anomaly_lead_ids = set(same_enquiry_anomaly_review['referral_lead_id_original'].dropna().astype('string'))
overlapping_lead_ids = sorted(suspected_lead_ids & same_enquiry_anomaly_lead_ids)

suspected_by_lead = suspected_bypass_review.groupby('booking_lead_id', dropna=False).agg(
    suspected_bypass_booking_count=('booking_id', 'nunique'),
    suspected_booking_ids=('booking_id', lambda values: ', '.join(sorted(set(values.dropna().astype(str))))),
    suspected_referral_enquiry_ids=('referral_enquiry_id', lambda values: ', '.join(sorted(set(values.dropna().astype(str))))),
).reset_index().rename(columns={'booking_lead_id': 'lead_id'})

same_anomaly_by_lead = same_enquiry_anomaly_review.groupby('referral_lead_id_original', dropna=False).agg(
    same_enquiry_anomaly_count=('referral_enquiry_id', 'nunique'),
    same_enquiry_referral_enquiry_ids=('referral_enquiry_id', lambda values: ', '.join(sorted(set(values.dropna().astype(str))))),
    same_enquiry_booking_ids=('booking_on_referral_enquiry_id', lambda values: ', '.join(sorted(set(values.dropna().astype(str))))),
).reset_index().rename(columns={'referral_lead_id_original': 'lead_id'})

overlap_by_lead = suspected_by_lead.merge(same_anomaly_by_lead, on='lead_id', how='inner')
overlap_by_lead = overlap_by_lead.loc[overlap_by_lead['lead_id'].isin(overlapping_lead_ids)].sort_values('lead_id')

# 2) Referral enquiry ID overlap
suspected_referral_enquiry_ids = set(suspected_bypass_review['referral_enquiry_id'].dropna().astype('string'))
same_anomaly_referral_enquiry_ids = set(same_enquiry_anomaly_review['referral_enquiry_id'].dropna().astype('string'))
overlapping_referral_enquiry_ids = sorted(suspected_referral_enquiry_ids & same_anomaly_referral_enquiry_ids)

overlap_by_referral_enquiry_id = pd.DataFrame({
    'referral_enquiry_id': overlapping_referral_enquiry_ids,
})
if not overlap_by_referral_enquiry_id.empty:
    overlap_by_referral_enquiry_id = overlap_by_referral_enquiry_id.merge(
        suspected_bypass_review[['referral_enquiry_id', 'booking_id', 'booking_lead_id', 'booking_project_name']],
        on='referral_enquiry_id',
        how='left'
    ).merge(
        same_enquiry_anomaly_review[['referral_enquiry_id', 'referral_lead_id_original', 'projectName', 'booking_on_referral_enquiry_id']],
        on='referral_enquiry_id',
        how='left',
        suffixes=('_suspected_bypass', '_same_enquiry')
    )

# 3) Strict booking / enquiry ID overlap
strict_overlap_rows = []

if {'booking_id'}.issubset(suspected_bypass_review.columns) and {'booking_on_referral_enquiry_id'}.issubset(same_enquiry_anomaly_review.columns):
    booking_id_overlap = sorted(
        set(suspected_bypass_review['booking_id'].dropna().astype('string'))
        & set(same_enquiry_anomaly_review['booking_on_referral_enquiry_id'].dropna().astype('string'))
    )
    for overlap_id in booking_id_overlap:
        strict_overlap_rows.append({
            'match_type': 'suspected_booking_id = same_enquiry_booking_id',
            'overlap_id': overlap_id,
            'note': 'The suspected bypass booking is the same booking recorded in the same-enquiry anomaly table.'
        })
else:
    booking_id_overlap = []

if {'enquiry_id'}.issubset(suspected_bypass_review.columns) and {'referral_enquiry_id'}.issubset(same_enquiry_anomaly_review.columns):
    enquiry_id_overlap = sorted(
        set(suspected_bypass_review['enquiry_id'].dropna().astype('string'))
        & set(same_enquiry_anomaly_review['referral_enquiry_id'].dropna().astype('string'))
    )
    for overlap_id in enquiry_id_overlap:
        strict_overlap_rows.append({
            'match_type': 'suspected_booking_enquiry_id = same_enquiry_referral_enquiry_id',
            'overlap_id': overlap_id,
            'note': 'The suspected bypass booking is linked to the same referral enquiry ID as a same-enquiry anomaly.'
        })
else:
    enquiry_id_overlap = []

overlap_by_booking_or_enquiry_id = pd.DataFrame(
    strict_overlap_rows,
    columns=['match_type', 'overlap_id', 'note']
)

lead_overlap_count = len(overlapping_lead_ids)
referral_enquiry_overlap_count = len(overlapping_referral_enquiry_ids)
strict_booking_or_enquiry_overlap_count = len(overlap_by_booking_or_enquiry_id)
combined_unique_leads_across_two_diagnostics = len(suspected_lead_ids | same_enquiry_anomaly_lead_ids)

if strict_booking_or_enquiry_overlap_count > 0 or referral_enquiry_overlap_count > 0:
    overlap_interpretation = 'Partially overlapping at strict record identifier level.'
elif lead_overlap_count > 0:
    overlap_interpretation = 'Partially overlapping at lead level only; no strict referral enquiry or booking/enquiry ID overlap was found.'
else:
    overlap_interpretation = 'No overlap found under the checked lead, referral enquiry, or booking/enquiry identifiers.'

overlap_summary = pd.DataFrame([
    {'metric': 'suspected_booking_level_bypass_cases', 'value': len(suspected_bypass_review), 'interpretation': 'Booking-centric suspected bypass cases.'},
    {'metric': 'same_enquiry_attribution_anomalies', 'value': len(same_enquiry_anomaly_review), 'interpretation': 'Referral-enquiry-level attribution anomalies.'},
    {'metric': 'overlapping_leads', 'value': lead_overlap_count, 'interpretation': 'Leads appearing in both diagnostic views.'},
    {'metric': 'overlapping_referral_enquiry_ids', 'value': referral_enquiry_overlap_count, 'interpretation': 'Same referral enquiry IDs appearing in both views.'},
    {'metric': 'strict_booking_or_enquiry_overlaps', 'value': strict_booking_or_enquiry_overlap_count, 'interpretation': 'Strict booking/enquiry identifier overlaps across both views.'},
    {'metric': 'combined_unique_leads_across_two_diagnostics', 'value': combined_unique_leads_across_two_diagnostics, 'interpretation': 'Unique leads affected across both diagnostic views after lead-level deduplication.'},
    {'metric': 'current_overlap_interpretation', 'value': overlap_interpretation, 'interpretation': 'Current reporting classification for the 4 and 28 findings.'},
])

overlap_by_lead.to_csv(OUTPUT_DIR / 'overlap_by_lead.csv', index=False)
overlap_by_referral_enquiry_id.to_csv(OUTPUT_DIR / 'overlap_by_referral_enquiry_id.csv', index=False)
overlap_by_booking_or_enquiry_id.to_csv(OUTPUT_DIR / 'overlap_by_booking_or_enquiry_id.csv', index=False)
overlap_summary.to_csv(OUTPUT_DIR / 'overlap_summary.csv', index=False)

print('Overlap summary:')
display(overlap_summary)
print('Lead-level overlap:')
display(overlap_by_lead)
print('Referral enquiry ID overlap:')
display(overlap_by_referral_enquiry_id)
print('Strict booking / enquiry ID overlap:')
display(overlap_by_booking_or_enquiry_id)


## How to Interpret the Two Findings

### Finding 1: 4 Suspected Booking-Level Bypass Candidates

This finding starts from the final booking records.

**Logic**

- Start with each booking.
- Look backward to check whether the same lead had any referral before the booking date.
- If a prior referral exists, match the booking to the most recent prior referral.
- Check whether the booking is marked as REFERRAL.
- If the booking is not marked as REFERRAL, flag it as a suspected bypass candidate.

**Business meaning**

This finding answers the question:

> Did this final booking have a prior referral that was not honored as REFERRAL?

This is the closest diagnostic route to the manager's commission-saving concern.

**Current numbers**

- 5,275 booking records.
- 20 bookings had an eligible prior referral.
- 16 of those 20 were honored as REFERRAL.
- 4 were not marked as REFERRAL and were flagged as suspected bypass candidates.

**Interpretation**

The 4 suspected bypass cases should be treated as high-confidence booking-level review cases under the conservative exact lead_id matching rule.

### Finding 2: 28 Same-Enquiry Attribution Anomalies

This finding starts from the referral enquiry records themselves.

**Logic**

- Start with referral enquiries.
- The enquiry itself is originally from a REFERRAL source.
- The same referral enquiry later produced a booking.
- However, the booking was not marked as REFERRAL.
- This is flagged as a same-enquiry attribution anomaly.

**Business meaning**

This finding answers the question:

> Did this referral enquiry itself convert into a booking, but fail to be honored as REFERRAL?

This is more like an internal attribution anomaly within the same referral enquiry, rather than a booking-centric backward match across all bookings.

**Current numbers**

- 2,734 own referral enquiries were honored.
- 28 own referral enquiries were not honored.
- These 28 anomalies involve only 7 leads.

**Interpretation**

The 28 same-enquiry anomalies should not be directly added to the 4 suspected booking-level bypass cases unless overlap checks confirm that they represent distinct cases. Because the 28 anomalies involve only 7 leads, they may include repeated anomaly records under a small number of leads.

### Why 28 Can Be Larger Than 4

The 28 anomalies are not calculated from the 20 booking-prior-referral matches. They come from a different diagnostic route.

The 4 suspected bypass cases use a stricter booking-centric route:

- Start from booking table.
- Match to prior referral using same lead_id.
- Require referral date to be before booking date.
- Use the latest prior referral.
- Require booking not marked as REFERRAL.

The 28 same-enquiry anomalies use a referral-enquiry-level route:

- Start from referral enquiry table.
- Check whether the referral enquiry itself has a booking.
- If that own booking is not honored as REFERRAL, flag it as an anomaly.

Therefore:

- The 4 cases are fewer because the rule is stricter and conservative.
- The 28 cases are more numerous because they check referral enquiry attribution directly and may include multiple enquiry-level records under the same lead.
- The two findings should be reported as separate diagnostic views until overlap is verified.

### Current Reporting Position

At this stage, I would report the findings as two separate diagnostic views. The booking-centric analysis identifies 4 high-confidence suspected bypass bookings under a conservative exact lead_id and latest-prior-referral matching rule. Separately, the same-enquiry attribution check identifies 28 referral-enquiry-level attribution anomalies across 7 leads. Before combining or counting these findings together, I ran an overlap check to determine whether they represent distinct cases or overlapping records viewed from different analytical routes.

**Overlap result:** the two findings partially overlap at the lead level only. One lead appears in both diagnostic views, but there is no overlap at the referral enquiry ID level and no strict booking/enquiry ID overlap. Therefore, the 4 and 28 should not be reported as the same records, and they also should not be casually added as 32 independent customer-level issues.


## 8. Summary Tables

These tables make the result easier to review: total suspected cases, projects affected, sources involved, and manual-review complexity.

In [ ]:
summary_by_case = booking_referral_match_cases.groupby('case_type', dropna=False).agg(
    booking_count=('booking_id', 'nunique'),
    lead_count=('booking_lead_id', 'nunique'),
    total_agreement_value=('agreementValue_num', 'sum'),
    same_project_count=('same_project', 'sum'),
    manual_review_count=('needs_manual_review', 'sum'),
    median_days_from_referral_to_booking=('days_between_referral_and_booking', 'median')
).reset_index()

summary_by_project = suspected_bypass_cases.groupby('booking_project_name', dropna=False).agg(
    suspected_booking_count=('booking_id', 'nunique'),
    suspected_lead_count=('booking_lead_id', 'nunique'),
    total_agreement_value=('agreementValue_num', 'sum'),
    same_project_count=('same_project', 'sum')
).reset_index().sort_values('suspected_booking_count', ascending=False)

summary_by_source = suspected_bypass_cases.groupby(['booking_source_norm', 'enquiry_source_norm'], dropna=False).agg(
    suspected_booking_count=('booking_id', 'nunique'),
    suspected_lead_count=('booking_lead_id', 'nunique'),
    total_agreement_value=('agreementValue_num', 'sum')
).reset_index().sort_values('suspected_booking_count', ascending=False)

manual_review_summary = booking_referral_match_cases.groupby(['case_type', 'needs_manual_review'], dropna=False).agg(
    booking_count=('booking_id', 'nunique'),
    lead_count=('booking_lead_id', 'nunique')
).reset_index()

referral_bypass_summary = pd.DataFrame([
    {'metric': 'referral_rows', 'value': len(referrals)},
    {'metric': 'booking_rows', 'value': len(bookings)},
    {'metric': 'unique_referral_leads', 'value': referrals['referral_lead_id'].nunique(dropna=True)},
    {'metric': 'unique_booking_leads', 'value': bookings['booking_lead_id'].nunique(dropna=True)},
    {'metric': 'bookings_with_any_prior_referral', 'value': booking_referral_match_cases['booking_id'].nunique()},
    {'metric': 'honored_referral_bookings_with_prior_referral', 'value': booking_referral_match_cases.loc[booking_referral_match_cases['case_type'].eq('Honored Referral Booking'), 'booking_id'].nunique()},
    {'metric': 'suspected_referral_bypass_bookings', 'value': suspected_bypass_cases['booking_id'].nunique()},
    {'metric': 'suspected_bypass_leads', 'value': suspected_bypass_cases['booking_lead_id'].nunique()},
    {'metric': 'suspected_bypass_total_agreement_value', 'value': suspected_bypass_cases['agreementValue_num'].sum()},
    {'metric': 'same_enquiry_attribution_anomaly_rows', 'value': len(same_enquiry_anomalies)},
    {'metric': 'same_enquiry_attribution_anomaly_leads', 'value': same_enquiry_anomalies['referral_lead_id_original'].nunique(dropna=True)},
    {'metric': 'overlapping_leads_between_two_diagnostics', 'value': lead_overlap_count},
    {'metric': 'overlapping_referral_enquiry_ids_between_two_diagnostics', 'value': referral_enquiry_overlap_count},
    {'metric': 'strict_booking_or_enquiry_overlaps_between_two_diagnostics', 'value': strict_booking_or_enquiry_overlap_count},
    {'metric': 'combined_unique_leads_across_two_diagnostics', 'value': combined_unique_leads_across_two_diagnostics},
])

summary_by_case.to_csv(OUTPUT_DIR / 'summary_by_case_type.csv', index=False)
summary_by_project.to_csv(OUTPUT_DIR / 'suspected_bypass_by_project.csv', index=False)
summary_by_source.to_csv(OUTPUT_DIR / 'suspected_bypass_by_source.csv', index=False)
manual_review_summary.to_csv(OUTPUT_DIR / 'manual_review_summary.csv', index=False)
referral_bypass_summary.to_csv(OUTPUT_DIR / 'referral_bypass_summary.csv', index=False)

print('Summary by case type:')
display(summary_by_case)
print('Suspected bypass by project:')
display(summary_by_project)
print('Suspected bypass by source:')
display(summary_by_source)
print('Manual review summary:')
display(manual_review_summary)
print('Overall summary:')
display(referral_bypass_summary)

## 9. Tableau / Review Export

This compact table is designed for quick review or Tableau import. It keeps one row per booking-prior-referral matched case and includes the fields most useful for filtering.

In [ ]:
tableau_review_export = booking_referral_match_cases.assign(
    booking_month=lambda df: df['booking_timeline_at'].dt.to_period('M').astype('string'),
    referral_month=lambda df: df['referral_submitted_at'].dt.to_period('M').astype('string'),
    agreement_value_million=lambda df: df['agreementValue_num'] / 1_000_000
)[[
    'case_type', 'needs_manual_review', 'booking_id', 'booking_month', 'booking_timeline_at',
    'booking_project_name', 'booking_source_norm', 'enquiry_source_norm', 'booking_lead_id',
    'booking_lead_name', 'agreementValue_num', 'agreement_value_million',
    'referral_enquiry_id', 'referral_month', 'referral_submitted_at', 'referral_project_name',
    'same_project', 'days_between_referral_and_booking', 'referrer_user_id', 'referrer_name',
    'prior_referral_count', 'prior_referrer_count', 'prior_project_count'
]].sort_values(['case_type', 'booking_timeline_at'])

tableau_review_export.to_csv(OUTPUT_DIR / 'tableau_referral_bypass_review_export.csv', index=False)

print(f'Tableau/review export rows: {len(tableau_review_export):,}')
display(tableau_review_export.head(20))

## 10. Questions to Confirm with Manager

1. Should each booking be matched to the latest prior referral, or should any historical prior referral qualify?
2. Should there be a maximum attribution window, such as 30, 60, or 90 days?
3. For same-enquiry anomalies, should these be treated as suspected bypass cases, attribution quality issues, or a separate review category?
4. If the 4 suspected bypass cases and 28 same-enquiry anomalies overlap at the lead level but not the booking/enquiry level, how should they be reported?
5. Should encrypted mobile/email exact matching be added for cases where lead_id is missing or inconsistent?
6. Should commission impact be estimated from agreementValue, or is there a separate commission table?
